In [ ]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import yaml
import numpy as np

def get_project_root():
    return Path().resolve().parent

# This file is the master mapping from city to province, the comprehensivre list with codes
# there are 514 unique districts/cities, and 38 unique provinces
df = pd.read_csv(get_project_root() / "data" / "mapping" / "master_2024.csv")

# These are the manually coded assessments of the index indicators (by Hana)
# otherwise known as Hana's dataset
# there are 509 unique cities here (5 missing are the DKI JAKARTA...)
policy_encoding = pd.read_csv(get_project_root() / "data" / "processed" / "policy_encoding.csv")

# maps from official (master) fullname to/from Hanas encoding
with open(get_project_root() / "data" / "mapping"/"city_mapping.yaml", "r") as f:
    city_mapping = yaml.safe_load(f)

city_mapping_reversed = {v: k for k, v in city_mapping.items()}

In [2]:
# create nested dict version of master mapping from master full name for quick reference
# dont need to run unless master changes
def create_tree_yaml(df):
    prov_tree = defaultdict(list)

    for _, row in df.iterrows():
        prov_tree[row["nama_prov"]].append(row["fullname"])

    output_path = get_project_root() / "data" / "mapping"
    output_path.mkdir(parents=True, exist_ok=True)
    file_path = output_path / "prov_tree.yaml"

    with open(file_path, "w", encoding="utf-8") as f:
        yaml.dump(
            prov_tree,
            f,
            allow_unicode=True,
            sort_keys=True,
            default_flow_style=False
        )

In [ ]:
# look at filenames andmissing files in directory


<ArrowStringArray>
[                               'ACEH_BIREUEN.txt',
                      'ACEH_SUBULUSSALAM_KOTA.txt',
                                 'BALI_BADUNG.txt',
                                 'BALI_BANGLI.txt',
                               'BALI_BULELENG.txt',
                                'BALI_TABANAN.txt',
               'BANTEN_TANGERANG_SELATAN_KOTA.txt',
                          'BENGKULU_KEPAHIANG.txt',
                      'BENGKULU_REJANG_LEBONG.txt',
                             'BENGKULU_SELUMA.txt',
 ...
          'SUMATERA_SELATAN_OGAN_KOMERING_ULU.txt',
       'SUMATERA_SELATAN_OGAN_KOMERING_ULU_v2.txt',
  'SUMATERA_SELATAN_OGAN_KOMERING_ULU_SELATAN.txt',
             'SUMATERA_SELATAN_PALEMBANG_KOTA.txt',
 'SUMATERA_SELATAN_PENUKAL_ABAB_LEMATANG_ILIR.txt',
            'SUMATERA_SELATAN_PRABUMULIH_KOTA.txt',
                    'SUMATERA_UTARA_BATU_BARA.txt',
          'SUMATERA_UTARA_LABUHANBATU_SELATAN.txt',
         'SUMATERA_UTARA_PADANGSIDIMPUAN

### Policy texts

In [11]:
# This is the raw data (policy) documents, not yet translated
base = Path.cwd().parent / "data" / "raw" / "localpolicies"
filenames = policy_encoding['filename'].dropna().unique()

txt_exists = np.array([(base / f).exists() for f in filenames])

pdf_exists = np.array([
    (base / Path(f).with_suffix(".pdf").name).exists()
    for f in filenames
])

print(f"Total expected files: {len(filenames)}")
print(f"TXT exists: {txt_exists.sum()}")
print(f"TXT missing but PDF exists: {((~txt_exists) & pdf_exists).sum()}")
print(f"Missing entirely: {((~txt_exists) & (~pdf_exists)).sum()}")

Total expected files: 198
TXT exists: 149
TXT missing but PDF exists: 6
Missing entirely: 43


In [ ]:
# TODO:

- manual overwrites (collate from email)

- translate raw files
- prompts yaml versions (v2)

- bring index logic across (recount domains)
- geospatial visualisations of Index/GDP/SIMFONI

- document (codes)

# Q for Hana:
['DKI Jakarta'] -> DKI JAKARTA:
- DKI JAKARTA KEPULAUAN SERIBU
- DKI JAKARTA JAKARTA SELATAN KOTA
- DKI JAKARTA JAKARTA TIMUR KOTA
- DKI JAKARTA JAKARTA PUSAT KOTA
- DKI JAKARTA JAKARTA BARAT KOTA
- DKI JAKARTA JAKARTA UTARA KOTA

MALUKU master has 11, 12 in Hana's dataset (extra: "Maluku Tenggara Barat")
"MALUKU KEPULAUAN TANIMBAR": "" **** renamed 2019
"MALUKU MALUKU TENGGARA": "Maluku Tenggara"
"MALUKU MALUKU TENGAH": 'Maluku Tengah'
"MALUKU BURU": "Buru"
"MALUKU KEPULAUAN ARU": "Kepulauan Aru"
"MALUKU SERAM BAGIAN BARAT": "Seram Bagian Barat"
"MALUKU SERAM BAGIAN TIMUR": "Seram Bagian Timur"
"MALUKU MALUKU BARAT DAYA": "Maluku Barat Daya"
"MALUKU BURU SELATAN": "Buru Selatan"
"MALUKU AMBON KOTA": "Kota Ambon"
"MALUKU TUAL KOTA": "Kota Tual"

SULAWESI BARAT master has 6, hanas dataset has 5 (missing: Mamuju Tengah)
"SULAWESI BARAT MAJENE": "Majene"
"SULAWESI BARAT POLEWALI MANDAR": "Polewali Mandar"
"SULAWESI BARAT MAMASA": "Mamasa"
"SULAWESI BARAT MAMUJU": "Mamuju"
"SULAWESI BARAT PASANGKAYU": "Mamuju Utara"
"SULAWESI BARAT MAMUJU TENGAH": "Mamuju Tengah" ***** wrong provice categorisation?

what happeend with updated documents?
